# MuJoCo GLFW Viewer
- Multiple cameras available

</p>

> ### GLFW EXAMPLE
> 1. get mujoco model data & initialize glfw
> 2. create glfw window
> 3. setup scene & camera, append to list
> 4. step & render through while loop
>

</p>

In [1]:
import mujoco

import numpy as np
import time
import os
import glfw

import matplotlib.pyplot as plt

In [2]:
# 1-1. get model & data
xml_path = '../asset/ur_scene.xml'
xml_abs_path = os.path.abspath(xml_path)

model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)

In [9]:
# 1-2. lists for multiple cameras & windows
windows = []
contexts = [] # for mujoco context
scenes = []
cameras = []
options = []
viewport=mujoco.MjrRect(0, 0, 0, 0)

if not glfw.init():
    sys.exit("couldn't initialize glfw")

In [10]:
# 1-3. create window and context

# 1-3-a. free camera

# create window
window = glfw.create_window(1280, 720, f"Camera: interactive", None, None)
# validate
if not window:
    glfw.terminate()
    raise RuntimeError("GLFW window creation failed")

# make glfw context
glfw.make_context_current(window)

# scene, camera, options
scene = mujoco.MjvScene(model, maxgeom=1000)
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
opt = mujoco.MjvOption()

# append to list
scenes.append(scene)
cameras.append(cam)
options.append(opt)
windows.append(window)
contexts.append(mujoco.MjrContext(model, mujoco.mjtFontScale.mjFONTSCALE_150))

In [11]:
# 1-3-b. fixed camera

# create window
window = glfw.create_window(640, 480, f"Camera: egocentric fixed", None, None)
if not window:
    glfw.terminate()
    raise RuntimeError("GLFW window creation failed")

# make glfw context
glfw.make_context_current(window)

# scene, camera, options
scene = mujoco.MjvScene(model, maxgeom=1000)
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FIXED
cam.fixedcamid = 0
opt = mujoco.MjvOption()

# append to list
scenes.append(scene)
cameras.append(cam)
options.append(opt)
windows.append(window)
contexts.append(mujoco.MjrContext(model, mujoco.mjtFontScale.mjFONTSCALE_150))


# 1-4. set rendering interval
render_interval = 0.1
last_render_time = time.time()

In [12]:
""" GLFW CALLBACK: CAMERA MOVEMENT """

# 2-1. initialize global variable
last_x, last_y = 0, 0
mouse_button = None

# 2-2. define glfw callbacks

def mouse_button_callback(window, button, action, mods):
    global mouse_button
    if action == glfw.PRESS:
        mouse_button = button
        # print("mouse pressed")
    elif action == glfw.RELEASE:
        mouse_button = None

def cursor_pos_callback(window, xpos, ypos):
    global last_x, last_y, mouse_button
    dx = (xpos - last_x)/1000
    dy = (ypos - last_y)/1000
    last_x, last_y = xpos, ypos
    # print(f"mouse position: {xpos}, {ypos}")

    if mouse_button is not None:
        action = {
            glfw.MOUSE_BUTTON_LEFT: mujoco.mjtMouse.mjMOUSE_ROTATE_H,
            glfw.MOUSE_BUTTON_RIGHT: mujoco.mjtMouse.mjMOUSE_MOVE_H,
            glfw.MOUSE_BUTTON_MIDDLE: mujoco.mjtMouse.mjMOUSE_ZOOM
        }.get(mouse_button, None)

        if action is not None:
            mujoco.mjv_moveCamera(model, action, dx, dy, scenes[0], cameras[0])
            # print("cam moved")

def scroll_callback(window, xoffset, yoffset):
    # Zoom camera with scroll wheel
    mujoco.mjv_moveCamera(model, mujoco.mjtMouse.mjMOUSE_ZOOM, 0.0, -yoffset/100, scenes[0], cameras[0])
    # print(f"offsets x:{xoffset}, y:{yoffset}")


# 2-3. set glfw callbacks
glfw.set_mouse_button_callback(windows[0], mouse_button_callback)
glfw.set_cursor_pos_callback(windows[0], cursor_pos_callback)
glfw.set_scroll_callback(windows[0], scroll_callback)

In [ ]:
""" SIMULATION LOOP """

# 3-1. generate two actions
globaol_height = 720
global_width = 1280
pos_1 = [0, -0.8, 2.5, 0, -0.3, 0]
pos_2 = [0.8, 1.3, 0.2, 0, -0.3, 0]
control_signal = pos_1
flag = 0
count = 0

init_time = time.time()

# while loop
while not any([glfw.window_should_close(win) for win in windows]):
    # make control signal

    if flag == 1 and time.time() - init_time > 5:
        control_signal = pos_2
        flag = 0
        init_time = time.time()
    elif flag == 0 and time.time() - init_time > 5:
        count=0
        control_signal = pos_1
        flag = 1
        init_time = time.time()

    # set input control
    data.ctrl = control_signal

    # step
    mujoco.mj_step(model, data)
    # print("steped")

    # determine frame rate
    current_time = time.time()

    if current_time - last_render_time >= render_interval:
        last_render_time = current_time

        for i, window in enumerate(windows):
            
            # print(f"making context for window {i}")
            # make context, set width & height
            glfw.make_context_current(window)
            width, height = glfw.get_framebuffer_size(window)
            global_width = width
            global_height = height
            viewport.width = width
            viewport.height = height

            # update scene & render
            mujoco.mjv_updateScene(model, data, options[i], None, cameras[i],
                                mujoco.mjtCatBit.mjCAT_ALL, scenes[i])
            mujoco.mjr_render(viewport, scenes[i], contexts[i])

            glfw.swap_buffers(window)

        glfw.poll_events()
        time.sleep(0.01)

# Clean up
glfw.terminate()

</p>

> ### GLFW VIEWER CLASS
> - Default free camera
> - Add camera: free (default), fixed, track 
> - render, close, alive-check

</p>

In [ ]:
""" MUJOCO VIEWER CLASS """

import sys
import glfw 
from functools import partial

class MUJOCOGLVIEWER():
    def __init__(self, model, data):

        # initialize with empty lists
        glfw.init()
        self.model = model
        self.data = data

        self.windows = []
        self.contexts = []
        self.scenes = []
        self.cameras = []
        self.options = []
        self.viewport=mujoco.MjrRect(0, 0, 0, 0)
        if not glfw.init():
            sys.exit("couldn't initialize glfw")
        self.last_x, self.last_y = 0, 0
        self.mouse_button = None
        # render interval
        self.render_interval = 0.01
        self.last_render_time = time.time()
        # add default camera
        self.add_cameras(camera_names=["default camera"], types = ['free'], sizes=[(1280, 720)])

    def add_cameras(self,camera_names, types, sizes):
        if len(camera_names) != len(sizes) or len(camera_names) != len(types):
            raise ValueError("Length of camera_names, sizes, and types must be the same")
        # adding mujoco camera
        fixed_idx = 0
        track_idx = 0
        for i, (name, size, type) in enumerate(zip(camera_names, sizes, types)):
            window = glfw.create_window(size[0], size[1], f"Camera: {name}", None, None)
            if not window:
                glfw.terminate()
                raise RuntimeError("GLFW window creation failed")
            glfw.make_context_current(window)
            # scene, camera, options
            scene = mujoco.MjvScene(self.model, maxgeom=1000)
            cam = mujoco.MjvCamera()
            if type == 'free':
                cam.type = mujoco.mjtCamera.mjCAMERA_FREE
                actual_cursor_pos_callback = partial(self.cursor_pos_callback, scene, cam)
                actual_scroll_callback = partial(self.scroll_callback, scene, cam)
                glfw.set_mouse_button_callback(window, self.mouse_button_callback)
                glfw.set_cursor_pos_callback(window, actual_cursor_pos_callback)
                glfw.set_scroll_callback(window, actual_scroll_callback) 
            elif type == 'fixed':
                cam.type = mujoco.mjtCamera.mjCAMERA_FIXED
                cam.fixedcamid = fixed_idx
                fixed_idx += 1
            elif type == 'track':
                cam.type = mujoco.mjtCamera.mjCAMERA_TRACKING
                cam.trackbodyid = track_idx
                track_idx += 1
            opt = mujoco.MjvOption()

            # append to list
            self.scenes.append(scene)
            self.cameras.append(cam)
            self.options.append(opt)
            self.windows.append(window)
            self.contexts.append(mujoco.MjrContext(self.model, mujoco.mjtFontScale.mjFONTSCALE_150))
                 
    
    def is_alive(self):
        if self.model.ncam + 1 < len(self.cameras):
            raise ValueError(f"Number of cameras in model ({self.model.ncam}) is less than the total number of cameras being added ({len(self.cameras)})")
        if any([glfw.window_should_close(win) for win in self.windows]):
            return False
        else:
            return True

    def close(self):
        for window in self.windows:
            glfw.destroy_window(window)
        glfw.terminate()

    def render(self):
        current_time = time.time()
        if current_time - self.last_render_time >= self.render_interval:
            self.last_render_time = current_time

            for i, window in enumerate(self.windows):
            
                glfw.make_context_current(window)
                width, height = glfw.get_framebuffer_size(window)
                self.viewport.width = width
                self.viewport.height = height

                # update scene & render
                mujoco.mjv_updateScene(self.model, self.data, self.options[i], None, self.cameras[i],
                                    mujoco.mjtCatBit.mjCAT_ALL, self.scenes[i])
                # kernel crashes here, mjv update scene
                mujoco.mjr_render(self.viewport, self.scenes[i], self.contexts[i])
                glfw.swap_buffers(window)
            glfw.poll_events()

    def mouse_button_callback(self, window, button, action, mods):
        if action == glfw.PRESS:
            self.mouse_button = button
        elif action == glfw.RELEASE:
            self.mouse_button = None

    def cursor_pos_callback(self, scene, camera, window, xpos, ypos):
        dx = (xpos - self.last_x)/1000
        dy = (ypos - self.last_y)/1000
        self.last_x, self.last_y = xpos, ypos

        if self.mouse_button is not None:
            action = {
                glfw.MOUSE_BUTTON_LEFT: mujoco.mjtMouse.mjMOUSE_ROTATE_H,
                glfw.MOUSE_BUTTON_RIGHT: mujoco.mjtMouse.mjMOUSE_MOVE_H,
                glfw.MOUSE_BUTTON_MIDDLE: mujoco.mjtMouse.mjMOUSE_ZOOM
            }.get(self.mouse_button, None)
            if action is not None:
                mujoco.mjv_moveCamera(self.model, action, dx, dy, scene, camera)

    def scroll_callback(self, scene, camera, window, xoffset, yoffset):
        # Zoom camera with scroll wheel
        mujoco.mjv_moveCamera(self.model, mujoco.mjtMouse.mjMOUSE_ZOOM, 0.0, -yoffset/100, scene, camera)
        # print(f"offsets x:{xoffset}, y:{yoffset}")




In [ ]:
print(f"Total number of cameras: {model.ncam}")
print(f"Mode of each cam: {model.cam_mode}") # 0: fixed, 1: track

viewer = MUJOCOGLVIEWER(model, data)
viewer.add_cameras(camera_names=["egocentric"], types = ['fixed'], sizes=[(640, 360)])
viewer.add_cameras(camera_names=["camera: tracking"], types = ['tracking'], sizes=[(640, 360)])

qpos_1 = np.array([0, -0.5, 0, -1.0, 0, 1.0])
qpos_2 = np.array([0, -1.0, 0, -1.5, 0, 1.5])
start_time = time.time()

while viewer.is_alive():
    if time.time() - start_time < 5:
        data.qpos[:] = qpos_1
    elif time.time() - start_time < 10:
        data.qpos[:] = qpos_2
    else:
        start_time = time.time() # reset start time to loop between qpos_1 and qpos_2 every 5 seconds

    mujoco.mj_forward(model, data)
    time.sleep(0.01)
    viewer.render()

viewer.close()

Total number of cameras: 2
Mode of each cam: [0 1]
